In [1]:
import requests
import brotli
import json
import re
import time
import datetime

In [2]:
url = 'https://api.skinport.com/v1/items'
response = requests.get(url, headers={"Accept-Encoding": "br"})
print("status:", response.status_code)


status: 200


In [3]:
if response.status_code == 200:
    ts = time.time()
    time_stamp = datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
    raw = brotli.decompress(response.content)   # manual decompress, bypassing requests
    data = json.loads(raw)
    print(len(data), "items")
    print(data[0])
else:
    print(response.text[:300])

error: brotli: decoder failed

In [ ]:
data[:30]

In [ ]:
#extract kv pairs i am interested in
extract_kv = ['market_hash_name', 'min_price', 'quantity']
extracted_list = []

for i in data:
    temp = {}
    for k in extract_kv:
        if k in i:
            temp[k] = i[k]
    extracted_list.append(temp)

In [ ]:
 #I need a separate field in the dict for the wear/stattrak(★)/souvenir/sticker(Gold,Holo,Foil,Glitter) of the item.


#Ok so I need to add fields, extract it from the market_hash_name, maybe add external general item dataset to crosscheck:



In [ ]:
for item in extracted_list:
    print(item['market_hash_name'])



In [ ]:
dupey = extracted_list
len(dupey)

In [ ]:
"""Item:
    raw_name: str            # the original market_hash_name
    item_name: str            # the base name with wear/quality stripped, e.g. "Talon Knife"
    item_type: str            # "weapon"  "knife"  "gloves"  "sticker"  "other"
    quality_tag: str | None   # "StatTrak/StatTrak™/★"  "Souvenir" | None  — mutually exclusive, weapons/knives only
    wear: str | None          # "Factory New"  ...  "Battle-Scarred" None for item_types that don't have wear
    sticker_finish: str | None  # "Foil"  "Holo"  "Gold"  "Glitter"  "Embroidered"  None — only populated when item_type == "sticker"
"""



for item in dupey:
    name = item['market_hash_name']
    item['ts'] = time_stamp
    item['venue'] = 'Skinport'
    if 'StatTrak' in name:
        item['quality_tag'] = 'StatTrak'
    elif 'Souvenir' in name:
        item['quality_tag'] = 'Souvenir'
    else:
        item['quality_tag'] = None


    if 'Gloves' in name:
        item['item_type'] = 'Gloves'
    elif 'Graffiti' in name:
        item['item_type'] = 'Graffiti'
    elif 'Charm' in name:
        item['item_type'] = 'Charm'
    elif '★' in name or 'Knife' in name and 'Gloves' not in name:
        item['item_type'] = 'Knife'
    elif 'Sticker Capsule' in name and not 'Sticker Slab' in name:
        item['item_type'] = 'Sticker Capsule'
    elif 'Sticker Slab' in name:
        item['item_type'] = 'Sticker Slab'
    elif 'Sticker' in name and not 'Sticker Slab' in name:
        item['item_type'] = 'Sticker'
    elif 'Patch |' in name:
        item['item_type'] = 'Patch'
    elif 'Music Kit' in name:
        item['item_type'] = 'Music Kit'
    else:
        item['item_type'] = 'Weapon'

    if 'Factory New' in name:
        item['wear'] = 'Factory New'
    elif 'Minimal Wear' in name:
        item['wear'] = 'Minimal Wear'
    elif 'Field-Tested' in name:
        item['wear'] = 'Field-Tested'
    elif 'Well-Worn' in name:
        item['wear'] = 'Well-Worn'
    elif 'Battle-Scarred' in name:
        item['wear'] = 'Battle-Scarred'
    else:
        item['wear'] = None


    if item['item_type'] == 'Sticker':
        if 'Foil' in name:
            item['sticker_finish'] = 'Foil'
        elif 'Holo' in name:
            item['sticker_finish'] = 'Holo'
        elif 'Gold' in name:
                item['sticker_finish'] = 'Gold'
        elif 'Glitter' in name:
            item['sticker_finish'] = 'Glitter'
        elif 'Embroidered' in name:
            item['sticker_finish'] = 'Embroidered'
        else:
            item['sticker_finish'] = None


In [ ]:
len(dupey)

In [ ]:
dupey_clean = []
for item in dupey:
    if item['item_type'] in ('Weapon', 'Knife', 'Gloves') and item['wear'] is not None:
        dupey_clean.append(item)
    elif item['item_type'] in ('Sticker', 'Sticker Capsule', 'Sticker Slab', 'Patch', 'Graffiti', 'Charm', 'Music Kit'):
        dupey_clean.append(item)
    else:
        continue

len(dupey_clean)

In [ ]:
for item in dupey_clean:
    word = item['market_hash_name']
    clean_text = word.replace("★ StatTrak™ ", "")
    clean_text = clean_text.replace("StatTrak™ ", "")
    clean_text = clean_text.replace("Souvenir ", "")
    clean_text = clean_text.replace(" Sticker Capsule", "")
    clean_text = clean_text.replace("★ ", "")
    clean_text = clean_text.replace("Sticker | ", "")
    clean_text = clean_text.replace("Music Kit | ", "")
    clean_text = re.sub(r"\s*\([^)]*\)", "", clean_text)
    item['market_hash_name'] = clean_text

len(dupey_clean)


In [ ]:
dupey